In [ ]:
import duckdb

In [ ]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [ ]:
df = con.execute("""
                SELECT * 
                FROM (
                    SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestão DESC) AS row 
                    FROM bronze_z0019
                    WHERE data_ingestão >= '2026-09-06'
                ) WHERE row = 1
                """).fetchdf()
df.head(10)

In [ ]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestão', 'row'])
df_final = df_final.rename(columns={"NATBR":"id"})
df_final = df_final.rename(columns={"MAKTX":"nm_produto"})
df_final = df_final.rename(columns={"WERKS":"id_categoria"})
df_final = df_final.rename(columns={"MAINS":"id_fornecedor"})
df_final = df_final.rename(columns={"LABST":"vl_preço"})
df_final.head(10)

In [ ]:
df_final.dtypes

In [ ]:
df2 = df_final
df2 = df2.astype(
    {
        'id': int,
        'nm_produto': str,
        'id_categoria': str,
        'id_fornecedor': int,
        'vl_preço': float
    }

)

df2.dtypes
#df_final.head(10)

In [ ]:
con.execute("""
CREATE TABLE IF NOT EXISTS produtos (
            id BIGINT,
            nm_produtos TEXT,
            id_categoria TEXT,
            id_fornecedor BIGINT,
            vl_preço FLOAT
            )
""")

In [ ]:
df2.head(10)

In [ ]:
con.execute("INSERT INTO produtos SELECT * FROM df2")

In [ ]:
df_resultado = con.execute("select * from produtos").fetchdf()
df_resultado.head(10)

In [ ]:
con.close()